# Notebook 01: Data preparation and preprocessing

This notebook is derived from **`mmr comprehensive.ipynb`**. It isolates everything needed to go from the **merged country–year file** to an **analysis-ready panel**: loading data, checking structure, standardizing identifiers, defining modeling features, applying a **log1p** transform to maternal mortality, optional **within-country median imputation** for missing predictors, and **saving** a single CSV for Notebooks 02–04.

**Run this notebook first.** Downstream notebooks read:

`../data/processed/country_year_modeling_panel_cleaned.csv`

**Inputs (first path that exists):** `../data/processed/country_year_modeling_dataset.csv` or `../data/country_year_modeling_dataset.csv`.


## Dataset shape and dtypes

Same checks as the original comprehensive notebook.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

CANDIDATES = [
    Path("../data/processed/country_year_modeling_dataset.csv"),
    Path("../data/country_year_modeling_dataset.csv"),
]
INPUT_PATH = next((p for p in CANDIDATES if p.exists()), None)
OUT_PATH = Path("../data/processed/country_year_modeling_panel_cleaned.csv")
if INPUT_PATH is None:
    raise FileNotFoundError("Place country_year_modeling_dataset.csv in ../data/processed/ or ../data/")

df = pd.read_csv(INPUT_PATH)
df.head()


In [ ]:
print("Shape:", df.shape)
print(df.info())
#5712 instances and 12 variables
#no missing values
#all numeric predictors
#ISO3 and country are categorical identifiers


## Identifiers, feature list, and `log_mmr`

Aligns with the comprehensive notebook: integer `year`, clean `iso3c`, define `features`, create `log_mmr` if absent.


In [ ]:
# Ensure year is int and iso3c clean
df["year"] = df["year"].astype(int)
df["iso3c"] = df["iso3c"].astype(str).str.strip().str.upper()

#because of the skewness, applied a log transformation to stabilize variance and improve model performance
#log_mmr = log(1 + MMR)
# If log_mmr isn't in the file, create it
if "log_mmr" not in df.columns:
    df["log_mmr"] = np.log1p(df["mmr"])

target = "log_mmr"

features = [
    "gdp_pc",
    "health_exp_gdp",
    "fertility",
    "skilled_births",
    "pm25",
    "heat_index35_days",
    "cooling_degree_days",
]

# Keep only features that exist
features = [c for c in features if c in df.columns]
print("Using features:", features)


## Imputation (recommended for a clean panel)

The original notebook imputed inside sklearn `Pipeline`s; here we optionally fill missing **numeric features** using **within-country medians**, then global medians, so EDA and correlation tables in Notebook 02 are well-defined.


In [ ]:
for col in features:
    df[col] = df.groupby("country", group_keys=False)[col].transform(
        lambda s: s.fillna(s.median())
    )
df[features] = df[features].fillna(df[features].median())
print("Missing in features after imputation:", int(df[features].isna().sum().sum()))


## Save preprocessed panel


In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH.resolve())
